# Laboratorio L6 - Esplorare i dati con i grafici

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessari: `pinguini.csv`; per l'ultima parte `tragitti_puliti_classe.csv` prodotto in L5 (in alternativa `tragitti_puliti.csv`, fornito con il corso).

Dati: Palmer Penguins, Horst AM, Hill AP, Gorman KB (2020), licenza CC0, https://allisonhorst.github.io/palmerpenguins/

## 0. Preparazione

- `matplotlib.pyplot`, importato come `plt`, è la libreria dei grafici
- `COLORI` associa a ogni specie un colore; `SPECIE` fissa l'ordine: la stessa specie ha sempre lo stesso colore in tutti i grafici

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

pinguini = pd.read_csv("pinguini.csv")
SPECIE = ["Adelie", "Chinstrap", "Gentoo"]
COLORI = {"Adelie": "#2a78d6", "Chinstrap": "#eb6834", "Gentoo": "#1baf7a"}
MARCATORI = {"Adelie": "o", "Chinstrap": "s", "Gentoo": "^"}

## 1. Grafico a barre per una variabile qualitativa

- `value_counts()`: conteggi per specie; `reindex(SPECIE)`: nell'ordine fissato
- `plt.bar(etichette, altezze, color=...)`: barre
- `plt.ylabel`, `plt.title`: etichetta dell'asse e titolo
- `plt.show()`: mostra il grafico

In [ ]:
conteggi = pinguini["specie"].value_counts().reindex(SPECIE)
plt.bar(SPECIE, conteggi, color=[COLORI[s] for s in SPECIE])
plt.ylabel("numero di pinguini")
plt.title("Pinguini per specie")
plt.show()

## 2. Istogramma (esercizio 1)

- `plt.hist(valori, bins=n)`: divide l'intervallo dei valori in `n` intervalli (classi) e disegna una barra per il numero di valori in ciascuno
- `dropna()`: toglie i valori mancanti, che `hist` non accetta

In [ ]:
plt.hist(pinguini["pinna_lunghezza_mm"].dropna(), bins=20, color="#2a78d6", edgecolor="white")
plt.xlabel("lunghezza della pinna (mm)")
plt.ylabel("numero di pinguini")
plt.show()

Modificare `bins` (5, 20, 60) e osservare come cambia la forma. Poi eseguire la cella seguente: un istogramma per specie, sovrapposti (`alpha` rende le barre semitrasparenti).

**Domanda**: l'istogramma di tutti i pinguini ha due "gobbe". Come si spiegano con l'istogramma per specie?

Risposta:

In [ ]:
for s in SPECIE:
    valori = pinguini[pinguini["specie"] == s]["pinna_lunghezza_mm"].dropna()
    plt.hist(valori, bins=15, alpha=0.6, color=COLORI[s], label=s)
plt.xlabel("lunghezza della pinna (mm)")
plt.ylabel("numero di pinguini")
plt.legend()
plt.show()

## 3. Diagramma a scatola (esercizio 1)

`plt.boxplot(lista_di_serie)`: un diagramma a scatola per ogni serie; `plt.xticks(posizioni, nomi)` scrive i nomi sotto i diagrammi (posizioni 1, 2, 3...). La scatola va da Q1 a Q3, la linea interna è la mediana, i baffi arrivano al valore più estremo entro 1,5 x IQR, i punti isolati sono valori anomali.

In [ ]:
dati_specie = [pinguini[pinguini["specie"] == s]["becco_lunghezza_mm"].dropna() for s in SPECIE]
plt.boxplot(dati_specie)
plt.xticks(range(1, len(SPECIE) + 1), SPECIE)
plt.ylabel("lunghezza del becco (mm)")
plt.show()

Completare: disegnare i diagrammi a scatola della profondità del becco (`becco_profondita_mm`) per specie.

**Domanda**: quale specie si distingue con la profondità del becco? Quale con la lunghezza?

Risposta:

In [ ]:
dati_specie = [pinguini[pinguini["specie"] == s][___].dropna() for s in SPECIE]
plt.boxplot(dati_specie)
plt.xticks(range(1, len(SPECIE) + 1), SPECIE)
plt.ylabel("profondità del becco (mm)")
plt.show()

## 4. Grafico a dispersione (esercizio 2)

`plt.scatter(x, y, ...)`: un punto per ogni pinguino, con coordinate date da due variabili. Un colore e un marcatore per specie; `label` e `plt.legend()` producono la legenda.

In [ ]:
def dispersione(var_x, var_y):
    for s in SPECIE:
        d = pinguini[pinguini["specie"] == s]
        plt.scatter(d[var_x], d[var_y], color=COLORI[s], marker=MARCATORI[s],
                    s=25, edgecolor="white", linewidth=0.5, label=s)
    plt.xlabel(var_x)
    plt.ylabel(var_y)
    plt.legend()
    plt.show()

dispersione("becco_lunghezza_mm", "pinna_lunghezza_mm")

Provare altre coppie di variabili, per esempio `dispersione("becco_lunghezza_mm", "becco_profondita_mm")`.

**Domanda**: quale coppia di variabili separa meglio le tre specie? Descrivere dove si trova ciascuna specie nel grafico scelto. (Questa osservazione sarà la base delle regole di classificazione di L7.)

Risposta:

In [ ]:
dispersione("becco_lunghezza_mm", "becco_profondita_mm")

## 5. Matrice dei grafici a dispersione e correlazione (esercizio 3)

- `pd.plotting.scatter_matrix`: un grafico a dispersione per ogni coppia di variabili, istogrammi sulla diagonale
- `dropna(subset=misure)`: tiene solo i pinguini con tutte e quattro le misure; `map(COLORI)`: sostituisce a ogni specie il suo colore
- `corr()`: tabella dei coefficienti di correlazione di Pearson tra le colonne numeriche, da -1 a 1

In [ ]:
misure = ["becco_lunghezza_mm", "becco_profondita_mm", "pinna_lunghezza_mm", "massa_g"]
completi = pinguini.dropna(subset=misure)          # righe con tutte le misure
colori_punti = completi["specie"].map(COLORI)     # colore di ogni punto secondo la specie
pd.plotting.scatter_matrix(completi[misure], c=colori_punti, figsize=(9, 9), s=12, hist_kwds={"color": "#8a8984"})
plt.show()

In [ ]:
pinguini[misure].corr().round(2)

Completare: assegnare a `r` la correlazione tra lunghezza della pinna e massa.

In [ ]:
r = pinguini["pinna_lunghezza_mm"].corr(pinguini[___])
print(round(r, 2))
controlla(round(r, 2) == 0.87)

**Domanda**: su tutti i pinguini la correlazione tra lunghezza e profondità del becco è negativa (circa -0,24). Calcolarla per ciascuna specie con la cella seguente. Che cosa si osserva? Come si spiega? (Suggerimento: guardare il grafico a dispersione del becco.)

Risposta:

In [ ]:
for s in SPECIE:
    d = pinguini[pinguini["specie"] == s]
    print(s, round(d["becco_lunghezza_mm"].corr(d["becco_profondita_mm"]), 2))

## 6. I tragitti (esercizio 4)

Caricare il file pulito prodotto in L5. Se non è disponibile, usare `tragitti_puliti.csv`.

In [ ]:
tragitti = pd.read_csv("tragitti_puliti_classe.csv")
MEZZI = ["piedi", "bici", "monopattino", "bus", "auto", "treno"]
print(tragitti["mezzo"].value_counts())

In [ ]:
dati_mezzo = [tragitti[tragitti["mezzo"] == m]["tempo_min"].dropna() for m in MEZZI]
plt.boxplot(dati_mezzo)
plt.xticks(range(1, len(MEZZI) + 1), MEZZI)
plt.ylabel("tempo (minuti)")
plt.title("Tempo di tragitto per mezzo")
plt.show()

In [ ]:
plt.scatter(tragitti["distanza_km"], tragitti["tempo_min"], s=20, color="#2a78d6")
plt.xlabel("distanza (km)")
plt.ylabel("tempo (minuti)")
plt.show()
print("correlazione:", round(tragitti["distanza_km"].corr(tragitti["tempo_min"]), 2))

**Domande**

- Il tempo cresce con la distanza? La relazione è forte?
- Colorare i punti per mezzo (esercizio 5): a parità di distanza, quali mezzi sono più veloci?

Risposte:

## 7. Approfondimento (esercizio 5)

Completare la funzione per disegnare distanza e tempo con un colore per ciascun mezzo, sul modello della funzione `dispersione`.

In [ ]:
for i, m in enumerate(MEZZI):
    d = tragitti[tragitti["mezzo"] == m]
    plt.scatter(___, ___, s=22, label=m, color=f"C{i}")
plt.xlabel("distanza (km)")
plt.ylabel("tempo (minuti)")
plt.legend()
plt.show()